In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spur_zoom_S5_20261021_160000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spur_zoom_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
caps = pd.read_csv(RESULT_CSV).rename(columns={"index": "sweep"})  # "index" clashes with itertuples
traces = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")).rename(columns={"index": "sweep"})
GRID = "#e4e3df"
SIDE = {1: "upper", -1: "lower", 0: "carrier"}


def save(fig, name):
    fig.update_layout(template="plotly_white", font=dict(size=16))
    fig.write_html(RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}.html"))
    fig.show()


caps[caps["step"] != "sequence"][["label", "f_vco", "f_ref", "total_div", "f_pad", "locked", "carrier_dbm", "sideband",
                                  "step", "span", "rbw", "peak_offset", "peak_dbc"]].round(3)

,label,f_vco,f_ref,total_div,f_pad,locked,carrier_dbm,sideband,step,span,rbw,peak_offset,peak_dbc
0,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,0,carrier,2000.0,10.0,0.0,-0.016
1,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,1,zoom1,1000000.0,1000.0,1948000.0,-26.565
2,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,1,zoom2,50000.0,100.0,1947500.0,-27.305
3,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,1,zoom3,2000.0,10.0,1947512.0,-32.625
4,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,-1,zoom1,1000000.0,1000.0,-1587000.0,-28.125
5,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,-1,zoom2,50000.0,100.0,-1587400.0,-30.073
6,c9 min9 max0 760MHz,7.600000e+08,5937500.0,8,95000000.0,1,0.064,-1,zoom3,2000.0,10.0,-1587360.0,-35.149
37,c9 min14 max0 1GHz,1.000000e+09,7812500.0,16,62500000.0,1,1.387,0,carrier,2000.0,10.0,0.0,-0.013
38,c9 min14 max0 1GHz,1.000000e+09,7812500.0,16,62500000.0,1,1.387,1,zoom1,1000000.0,1000.0,1948000.0,-48.678
39,c9 min14 max0 1GHz,1.000000e+09,7812500.0,16,62500000.0,1,1.387,1,zoom2,50000.0,100.0,1947550.0,-47.661


In [3]:
# Per point: the carrier and every zoom on both sidebands, level relative to the carrier against the offset from the
# carrier (kHz; the lower sideband mirrored to positive offsets so both read the same). One figure per capture.
for (label, step, side), c in caps[caps["step"] != "sequence"].groupby(["label", "step", "sideband"], sort=False):
    r = c.iloc[0]
    t = traces[(traces["label"] == label) & (traces["step"] == step) & (traces["sideband"] == side)]
    offset = (t["freq"] - r["f_carrier"]) * (side if side else 1) / 1e3
    fig = go.Figure()
    fig.add_scatter(x=offset, y=t["level_dbm"] - r["carrier_dbm"], mode="lines", line=dict(width=1.5),
                    hovertemplate="%{x:.4f} kHz: %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"{label}: {SIDE[side]} {step} (span {r['span']:g} Hz, RBW {r['rbw']:g} Hz)", height=500)
    fig.update_xaxes(title="offset from the carrier [kHz]", gridcolor=GRID, exponentformat="none")
    fig.update_yaxes(title=f"level [dBc in {r['rbw']:g} Hz]", gridcolor=GRID)
    save(fig, f"{label.replace(' ', '_')}_{SIDE[side]}_{step}")

In [ ]:
# Sequence of single sweeps around the upper spur per point: the spur's offset from the carrier per sweep (does it
# move?), and all sweeps on top of each other (light: first, dark: last). One figure each.
from plotly.colors import sample_colorscale

for label, c in caps[caps["step"] == "sequence"].groupby("label", sort=False):
    fig = go.Figure()
    fig.add_scatter(x=c["sweep"], y=c["peak_offset"] / 1e3, mode="lines+markers", marker=dict(size=8),
                    hovertemplate="sweep %{x}: %{y:.3f} kHz<extra></extra>")
    fig.update_layout(title=f"{label}: spur offset per single sweep", height=450)
    fig.update_xaxes(title="sweep", gridcolor=GRID)
    fig.update_yaxes(title="spur offset from the carrier [kHz]", gridcolor=GRID, exponentformat="none")
    save(fig, f"{label.replace(' ', '_')}_sequence_offset")

    fig = go.Figure()
    t_all = traces[(traces["label"] == label) & (traces["step"] == "sequence")]
    for r in c.itertuples():
        t = t_all[t_all["sweep"] == r.sweep]
        fig.add_scatter(x=(t["freq"] - r.f_carrier) / 1e3, y=t["level_dbm"] - r.carrier_dbm, mode="lines",
                        showlegend=False, line=dict(width=1, color=sample_colorscale("Blues", [0.25 + 0.75 * r.sweep
                                                                                              / len(c)])[0]),
                        hovertemplate=f"sweep {r.sweep}" + "<br>%{x:.3f} kHz: %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"{label}: single sweeps around the spur", height=500)
    fig.update_xaxes(title="offset from the carrier [kHz]", gridcolor=GRID, exponentformat="none")
    fig.update_yaxes(title=f"level [dBc in {c['rbw'].iloc[0]:g} Hz]", gridcolor=GRID)
    save(fig, f"{label.replace(' ', '_')}_sequence_sweeps")

: 